In [1]:
import numpy as np
import pandas as pd
import requests
import io

In [2]:
# 자기 상대 파일 경로로 변경해도 됨
df = pd.read_excel('data\서울특별시 모기예보제 정보.xlsx')
df.head()

,mosquito_date,mosquito_value_house,mosquito_value_park
0,2016-05-01,254.4,254.4
1,2016-05-02,273.5,273.5
2,2016-05-03,304.0,304.0
3,2016-05-04,256.2,256.2
4,2016-05-05,243.8,243.8


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1849 entries, 0 to 1848
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   mosquito_date         1849 non-null   object 
 1   mosquito_value_house  1849 non-null   float64
 2   mosquito_value_park   1849 non-null   float64
dtypes: float64(2), object(1)
memory usage: 43.5+ KB


In [4]:
# 날짜 데이터 타입 변환 및 2020기준 후의 데이터만 사용, 인덱싱 초기화
df['mosquito_date'] = pd.to_datetime(df['mosquito_date'])
df = df[df['mosquito_date'] >= '2020-01-01']
df = df.reset_index(drop=True)
df.head()

,mosquito_date,mosquito_value_house,mosquito_value_park
0,2020-05-01,32.6,47.0
1,2020-05-02,36.6,52.7
2,2020-05-03,43.0,62.0
3,2020-05-04,43.2,62.3
4,2020-05-05,46.7,67.3


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1127 entries, 0 to 1126
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   mosquito_date         1127 non-null   datetime64[ns]
 1   mosquito_value_house  1127 non-null   float64       
 2   mosquito_value_park   1127 non-null   float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 26.5 KB


In [ ]:
def format_date_for_api(df):
    # 1. 안전하게 날짜(datetime) 타입으로 변환
    df['mosquito_date'] = pd.to_datetime(df['mosquito_date'])
    
    # 2. API가 요구하는 'YYYYMMDD0900' 형식의 문자열로 변환하여 새 컬럼 생성
    df['tm'] = df['mosquito_date'].dt.strftime('%Y%m%d')
    
    return df

def fetch_seoul_asos(df, auth_key):
    df = format_date_for_api(df)
    raw_data_list = []
    
    # 중복 호출을 막기 위해 고유한 날짜(tm)만 반복
    for tm in df['tm'].unique():
        # stn=108 (서울), help=0 (불필요한 설명줄 제거)
        # print(tm)
        url = f"https://apihub.kma.go.kr/api/typ01/url/kma_sfcdd.php?tm={tm}&stn=108&help=0&authKey={auth_key}"
        
        response = requests.get(url)
        
        # response 상태 확인용 print 출력문
        # print(response.status_code)
        
        # 응답이 정상(200)일 때만 텍스트 데이터 저장
        if response.status_code == 200:
            raw_data_list.append(response.text)
            
    return raw_data_list

def parse_kma_text(response_text):
    try:
        # 1. 핵심 변경 포인트: sep=',' (콤마 구분자로 변경)
        df_kma = pd.read_csv(io.StringIO(response_text), sep=',', comment='#', header=None)
        
        # (선택) 데이터 끝에 콤마가 있어서 생기는 의미 없는 빈 열(NaN) 제거
        df_kma = df_kma.dropna(axis=1, how='all')
        
    except pd.errors.EmptyDataError:
        return pd.DataFrame() # 데이터가 없는 날은 빈 데이터프레임 반환

    # data frame 확인용 print 출력문
    # print(df_kma)
    
    # 2. 필요한 데이터 열(인덱스) 추출
    # 이미지의 콤마 순서를 세어보면 대략 0:날짜, 1:지점, 10:평균기온, 17:평균습도, 37:일강수량 입니다.
    # (실제 컬럼 위치가 맞는지 한 번 더 확인해 주세요)
    df_extracted = df_kma[[0, 1, 10, 17, 37]]
    df_extracted.columns = ['tm', '지점', '평균기온', '평균습도', '일강수량']
    
    return df_extracted

In [7]:
# 본인 인증키로 변경 부탁드립니다
auth_key = 'l_38MrcRRpm9_DK3EbaZwg'

weather_list = fetch_seoul_asos(df.head(), auth_key)

mos_data_list = []

for item in weather_list:
    parse_kma_text(item)

C:\Users\chlal\AppData\Local\Temp\ipykernel_26256\1979358178.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['mosquito_date'] = pd.to_datetime(df['mosquito_date'])
C:\Users\chlal\AppData\Local\Temp\ipykernel_26256\1979358178.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tm'] = df['mosquito_date'].dt.strftime('%Y%m%d')


200
200
200
200
200
         0    1    2     3   4    5     6   7    8     9   ...  46   47  48  \
0  20200501  108  2.7  2328  29  5.5  1726  29  9.9  1752  ...  -9 -9.0  -9   

    49  50    51    52    53    54    55  
0 -9.0  -9  14.9  13.1  12.6  12.8  13.4  

[1 rows x 56 columns]
         0    1    2     3   4    5    6   7    8     9   ...  46   47  48  \
0  20200502  108  2.3  1985  23  4.6  111  29  7.6  1437  ...  -9 -9.0  -9   

    49  50    51    52    53    54    55  
0 -9.0  -9  15.8  13.4  12.7  12.8  13.4  

[1 rows x 56 columns]
         0    1    2     3   4    5     6   7    8     9   ...  46   47  48  \
0  20200503  108  2.0  1705  16  3.9  1242  14  7.2  1224  ...  -9 -9.0  -9   

    49  50    51    52    53    54    55  
0 -9.0  -9  16.2  13.7  12.9  12.9  13.4  

[1 rows x 56 columns]
         0    1    2     3   4    5     6   7    8     9   ...  46   47  48  \
0  20200504  108  2.7  2310  27  4.6  1247  27  8.7  1317  ...  -9 -9.0  -9   

    49  50    51   